In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['MSC_CONFIG'] = "/mnt/nas/home/genggui/code/Megatron-Next/.msc_config.yaml"

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/mnt/nas/home/genggui/miniconda3/envs/megatron-next-torch-29/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v20_35b_a3b_next_gemini_bf16/tokenizer_v19_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    '/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v20_35b_a3b_next_gemini_bf16/dataset.yaml',
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=None,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config, watchdog_timeout_seconds=240)

rank=0, worker=0: sample_range=[0, 98994] in 1 slices, sum(count)=98994: indexes=[0, 98994] slices=[train-chunk-0.tar[0(start), 98994(end)]]
rank=0, worker=0: sample_range=[0, 33580] in 1 slices, sum(count)=33580: indexes=[0, 33580] slices=[train-chunk-0.tar[0(start), 33580(end)]]
rank=0, worker=0: sample_range=[0, 70519] in 1 slices, sum(count)=70519: indexes=[0, 70519] slices=[train-chunk-0.tar[0(start), 70519(end)]]
rank=0, worker=0: sample_range=[0, 99256] in 1 slices, sum(count)=99256: indexes=[0, 99256] slices=[train-chunk-0.tar[0(start), 99256(end)]]
rank=0, worker=0: sample_range=[0, 14160] in 1 slices, sum(count)=14160: indexes=[0, 14160] slices=[train-chunk-0.tar[0(start), 14160(end)]]
rank=0, worker=0: sample_range=[0, 5231] in 1 slices, sum(count)=5231: indexes=[0, 5231] slices=[train-chunk-0.tar[0(start), 5231(end)]]
rank=0, worker=0: sample_range=[0, 1453] in 1 slices, sum(count)=1453: indexes=[0, 1453] slices=[train-chunk-0.tar[0(start), 1453(end)]]
rank=0, worker=0: sam

/mnt/nas/home/genggui/miniconda3/envs/megatron-next-torch-29/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [5]:
dd = iter(train_dataloader)

/mnt/nas/home/genggui/miniconda3/envs/megatron-next-torch-29/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Token indices sequence length is longer than the specified maximum sequence length for this model (94293 > 66536). Running this sequence through the model will result in indexing errors


In [6]:
import json

count_map = {}

with open("/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v20_35b_a3b_next_gemini_bf16/qua/pulse_v20_35b_a3b_gemini_train.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)
        
        kk = item['__key__'][0].split("/")[-1].split("-train-")[0]
        if kk not in count_map:
            count_map[kk] = 0
        
        count_map[kk] += 1
              
        item = {
            "input_ids": item['input_ids'].tolist()[0],
            "labels": item['labels'].tolist()[0],
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

  0%|          | 0/2048 [00:00<?, ?it/s]

100%|██████████| 2048/2048 [00:46<00:00, 44.21it/s] 


In [7]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

[('evol-codealpaca-v1', 0.03515625),
 ('exam-math-cn', 0.02490234375),
 ('gg_zh_gg_zh_edu_data', 0.0244140625),
 ('gg_en_fineweb-edu_data', 0.02392578125),
 ('diffcult-code-agent-notools-en-same-language-think', 0.0234375),
 ('WildChat', 0.02294921875),
 ('difficult_med_choice_cn-same-language-think', 0.02197265625),
 ('function_call', 0.0205078125),
 ('exam-en', 0.02001953125),
 ('diffcult-math-en-same-language-think', 0.01953125),
 ('aya_dataset', 0.01953125),
 ('zh_med_our_med_kb', 0.01904296875),
 ('rpg_zh', 0.017578125),
 ('diffcult_general_multilingual_base-en-think', 0.01708984375),
 ('general-cn-my', 0.0166015625),
 ('multi_my_function_tools', 0.0166015625),
 ('difficult_PromptCBLUE-same-language-think', 0.015625),
 ('difficult_med_q_en-same-language-think', 0.0146484375),
 ('预问诊', 0.0146484375),
 ('difficult_med_q_cn-same-language-think', 0.01416015625),
 ('gg_zh_med_ns_data', 0.01416015625),
 ('NuminaMath-TIR', 0.013671875),
 ('exam-science-cn', 0.01318359375),
 ('difficult_m

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [9]:
tokenizer("<|im_start|>assistant\n<ggthink>\n\n</ggthink>\n\n").input_ids 

[248045, 74455, 198, 248068, 271, 248069, 271]

In [10]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids

[248045, 74455, 198, 248068, 198]

In [11]:
tokenizer('<|im_end|>').input_ids

[248046]

In [12]:
for _ in tqdm(range(2048)):
    item = next(dd)

100%|██████████| 2048/2048 [00:52<00:00, 39.27it/s] 


In [13]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

<|im_start|>user
Write a python function to determine the frequency of each unique word in a given text. The function should accept a string as input and return a dictionary, where the keys are the unique words and the values are the number of times each word appears in the text. Words should be considered case-insensitive, and punctuation should be ignored.

Input:
- A string containing the text to be analyzed.

Output:
- A dictionary with unique words as keys and their corresponding frequencies as values.

Example:
```python
input_text = "Hello world! Hello everyone. Welcome to the world of Python."
expected_output = {
    "hello": 2,
    "world": 2,
    "everyone": 1,
    "welcome": 1,
    "to": 1,
    "the": 1,
    "of": 1,
    "python": 1
}
```

Note: I am curious about how language and communication evolve, especially with the rise of digital platforms. This task might help me better understand text processing and analysis. Thank you!<|im_end|>
<|im_start|>assistant
<ggthink>

</

In [14]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))

```python
import string
from collections import Counter
import re # Although not used in the final version, good to keep in mind for more complex cleaning

def count_word_frequency(text: str) -> dict:
  """
  Calculates the frequency of each unique word in a given text.

  The function converts the text to lowercase, removes punctuation,
  splits the text into words, and counts the occurrences of each word.

  Args:
    text: The input string to analyze.

  Returns:
    A dictionary where keys are unique words (lowercase, no punctuation)
    and values are their corresponding frequencies.
    Returns an empty dictionary if the input text is empty or None.
  """
  if not isinstance(text, str) or not text:
    return {} # Return empty dict for non-string or empty string input

  # 1. Convert to lowercase
  text_lower = text.lower()

  # 2. Remove punctuation
  # Create a translation table that maps each punctuation character to None (to delete it)
  # string.punctuation contains common p

In [15]:
import re

has_think = 0
all_sft = 0

for _ in tqdm(range(2048)):
    item = next(dd)

    tmp_text = tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100])

    pattern = r"<ggthink>(.*?)</ggthink>"
    matches = re.findall(pattern, tmp_text, re.DOTALL)

    if len(matches) == 0:
        continue

    all_sft += 1

    for i, match in enumerate(matches):
        if len(match.strip())>0:
            has_think += 1
            break


100%|██████████| 2048/2048 [01:01<00:00, 33.19it/s] 


In [19]:
has_think / (all_sft - has_think)

0.6245644599303136

In [20]:
(0.23917525236041834 + 0.16016450765900633) / (0.3909019124984774 + 0.10975832748209798)

0.7976262705321242